<a href="https://colab.research.google.com/github/beans0l/likelion-mju-ai/blob/main/week03/%EC%9D%B4%EB%B3%91%ED%97%8C/week03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

torch.manual_seed('60222509')

!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 데이터셋 및 데이터로더 클래스 정의
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 전체 텍스트 토큰화
        token_ids = tokenizer.encode(txt)

        # 슬라이딩 윈도우를 사용하여 데이터 분할
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

batch_size = 1
output_dim = 8
context_length = 4
stride = 4

# 데이터로더 생성
dataloader = create_dataloader_v1(
    text,
    batch_size=batch_size,
    max_length=context_length,
    stride=stride,
    shuffle=False
)

# 첫 번째 배치 데이터 추출
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

#초기화
tokenizer = tiktoken.get_encoding("gpt2")

# 임베딩 층 구현 및 결합
vocab_size = tokenizer.n_vocab

# 토큰 임베딩 층 생성
token_embedding_layer = nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)

# 위치 임베딩 층 생성 (절대 위치 인코딩)
pos_embedding_layer = nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))

# 최종 입력 임베딩 = 토큰 임베딩 + 위치 임베딩
input_embeddings = token_embeddings + pos_embeddings


print("### 첫 번째 배치 입력 임베딩 결과 ###")
print(f"입력 텐서 형태 (Batch, Context, Embedding): {input_embeddings.shape}")
print("\n첫 번째 배치의 입력 임베딩 값:")
print(input_embeddings)





--2026-05-10 12:31:29--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-05-10 12:31:29 (20.4 MB/s) - ‘input.txt.1’ saved [1115394/1115394]

### 첫 번째 배치 입력 임베딩 결과 ###
입력 텐서 형태 (Batch, Context, Embedding): torch.Size([1, 4, 8])

첫 번째 배치의 입력 임베딩 값:
tensor([[[-2.1302, -0.0072, -0.0866, -1.3949, -0.4172, -2.5380, -1.4495,
          -0.5377],
         [ 0.0778, -1.2035,  0.7323, -1.3311, -0.4650,  0.9223,  1.7115,
           2.1531],
         [ 0.8896,  1.0025,  1.0857, -0.3119, -2.6641,  2.8436, -0.9362,
          -1.3074],
         [ 1